# Your first compiled Python loop

**Core notebook, about 25 minutes.** We will measure a pure-Python loop, compile the same function with Numba, verify that both answers match, and benchmark steady-state execution.

The important habit is: **baseline, compile, verify, warm up, then time.**

In [ ]:
import numpy as np
import numba
from numba import njit

print("NumPy", np.__version__)
print("Numba", numba.__version__)

## 1. Establish the baseline

This function contains an explicit loop. It is intentionally written as clear Python before we optimize it.

In [ ]:
def diagonal_trace_python(a):
    trace = 0.0
    for i in range(a.shape[0]):
        trace += np.tanh(a[i, i])
    return trace

x = np.arange(1_000_000, dtype=np.float64).reshape(1000, 1000)
baseline = diagonal_trace_python(x)
python_time = %timeit -o -n 10 -r 3 diagonal_trace_python(x)

## 2. Compile, verify, and warm up

Since Numba 0.59, bare `jit` defaults to nopython mode. `njit` is the explicit alias for `jit(nopython=True)`, and this lesson uses it to keep that intent visible. Compilation occurs on the first call for a new input signature, so the first call must not be included in the steady-state benchmark.

In [ ]:
diagonal_trace_numba = njit(diagonal_trace_python)

# First call: compile this float64, two-dimensional signature.
compiled_result = diagonal_trace_numba(x)

# Correctness comes before speed.
np.testing.assert_allclose(compiled_result, baseline, rtol=1e-12)

# This timing now measures steady-state execution.
numba_time = %timeit -o -n 10 -r 3 diagonal_trace_numba(x)
print(f"Steady-state speedup: {python_time.average / numba_time.average:.1f}x")
print("Compiled signatures:", diagonal_trace_numba.signatures)

## 3. Compare with vectorized NumPy

Numba is not a replacement for clear NumPy. If an operation already vectorizes cleanly, compare both implementations.

In [ ]:
def diagonal_trace_numpy(a):
    return np.tanh(np.diagonal(a)).sum()

numpy_result = diagonal_trace_numpy(x)
np.testing.assert_allclose(numpy_result, baseline, rtol=1e-12)
numpy_time = %timeit -o -n 10 -r 3 diagonal_trace_numpy(x)

print(f"Python: {python_time.average * 1e3:.3f} ms")
print(f"Numba:  {numba_time.average * 1e3:.3f} ms")
print(f"NumPy:  {numpy_time.average * 1e3:.3f} ms")

## Your turn: sum of squares

**12 minutes.** Write a function that computes the sum of squares using a Python `for` loop. Then:

1. Compile it with `njit`.
2. Check it against `np.sum(values ** 2)`.
3. Warm up before timing.
4. Benchmark Numba and NumPy.
5. Explain the result in one sentence.

```python
def sum_of_squares(values):
    total = 0.0
    # Add your loop.
    return total
```

Blue sticky note means you need help. Yellow means you are ready to discuss.

In [ ]:
# Write and test your solution here before opening the solution cell below.

<details><summary>Solution and discussion</summary>

Run the next cell only after attempting the exercise. A simple reduction may be as fast or faster in NumPy. Numba is especially useful when a loop combines several operations, branches, or a pattern that does not vectorize cleanly.

</details>

In [ ]:
def sum_of_squares_python(values):
    total = 0.0
    for value in values:
        total += value * value
    return total

sum_of_squares_numba = njit(sum_of_squares_python)
rng = np.random.default_rng(2026)
values = rng.random(1_000_000)

expected = np.sum(values ** 2)
actual = sum_of_squares_numba(values)  # compile and warm up
np.testing.assert_allclose(actual, expected, rtol=1e-12)

exercise_numba_time = %timeit -o -n 10 -r 3 sum_of_squares_numba(values)
exercise_numpy_time = %timeit -o -n 10 -r 3 np.sum(values ** 2)
print(f"Numba: {exercise_numba_time.average * 1e6:.1f} us")
print(f"NumPy: {exercise_numpy_time.average * 1e6:.1f} us")

## Takeaway

- Use a profile or baseline timing to identify a real bottleneck.
- Prefer clear NumPy when it already expresses the operation efficiently.
- Try Numba for numerical loops that are difficult to vectorize.
- Verify the answer and exclude compilation from steady-state timing.

Optional next step: [`1_numpy.ipynb`](1_numpy.ipynb).